In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import os
import warnings
warnings.filterwarnings('ignore')

# load and prepare data (exactly like the main script)
from src.features import engineer_features
df_raw = pd.read_csv("data/SPY_daily.csv", index_col=0, parse_dates=True)
df = engineer_features(df_raw)

features = ['log_return', 'volatility_20d', 'z_score_20d']
X = df[features]
y = df['target_1d']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
train_data = lgb.Dataset(X_train, label=y_train)

# our artificial improvement steps for the research diary
steps = [
    {
        "name": "Step 1: Naive Baseline",
        "params": {'learning_rate': 0.1, 'num_leaves': 31, 'verbose': -1, 'random_state': 42},
        "reason": "baseline: model is too aggressive\nand memorizes market noise\n(overfitting).",
        "annotation_offset": (-0.1, -0.005)
    },
    {
        "name": "Step 2: Depth Regularization",
        "params": {'learning_rate': 0.05, 'num_leaves': 10, 'min_data_in_leaf': 50, 'verbose': -1, 'random_state': 42},
        "reason": "fix: shallower trees (leaves=10) &\nmin-data limits force the model\nto learn only robust patterns.",
        "annotation_offset": (-0.2, 0.005)
    },
    {
        "name": "Step 3: The Quant Sweet Spot",
        "params": {'learning_rate': 0.01, 'num_leaves': 7, 'feature_fraction': 0.8, 'verbose': -1, 'random_state': 42},
        "reason": "final: very slow learning rate (0.01)\n+ feature sampling (0.8) heavily\ndecorrelates the trees in LightGBM.",
        "annotation_offset": (-0.3, -0.008)
    }
]

# train models and collect results
results_auc = []
labels = []

print("=== Model Evolution Training ===\n")
for step in steps:
    # train with a fixed number of boost rounds for fair comparison
    model = lgb.train(step["params"], train_data, num_boost_round=100)
    y_proba = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)
    
    results_auc.append(auc)
    labels.append(step["name"])
    print(f"{step['name']} -> Out-of-Sample AUC: {auc:.4f}")

# create the visual story plot
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 7))

# draw the line and data points
ax.plot(labels, results_auc, marker='o', markersize=12, linewidth=3, color='#2c3e50', zorder=3)

# write explanations directly next to the points
for i, step in enumerate(steps):
    # format the parameter box
    param_text = "\n".join([f"{k}: {v}" for k, v in step["params"].items() if k not in ['verbose', 'random_state']])
    ax.annotate(
        f"[{param_text}]\n\n{step['reason']}", 
        (i, results_auc[i]),
        xytext=(i + step["annotation_offset"][0], results_auc[i] + step["annotation_offset"][1]),
        bbox=dict(boxstyle="round,pad=0.5", fc="#ecf0f1", ec="#bdc3c7", lw=1.5, alpha=0.9),
        fontsize=9,
        arrowprops=dict(arrowstyle="->", color="#7f8c8d", lw=1.5)
    )

ax.set_ylim(min(results_auc) - 0.01, max(results_auc) + 0.015)
ax.set_ylabel('Out-of-Sample AUC Score', fontsize=12)
ax.set_title('Model Evolution Diary: Regularization against Market Noise', fontsize=16, pad=20)
ax.grid(True, linestyle='--', alpha=0.6, zorder=0)

plt.tight_layout()
plt.show()